

***

# ✅ **What Are Catalogs in Azure Databricks?**

***

## **1. Definition**

A **catalog** is the **top-level container** in Unity Catalog’s data governance model.  
It’s the **first layer** in the three-level namespace:

    catalog.schema.table

*   **Catalog** → Groups schemas.
*   **Schema** → Groups tables, views, volumes, models, and functions.
*   **Tables/Volumes** → Actual data objects.

Think of a **catalog as a big folder** that organizes all your data objects logically.

***

## **2. Why Does It Matter?**

*   **Data Organization**: Keeps your data structured and easy to manage.
*   **Security**: Catalog-level permissions control access to everything inside.
*   **Isolation**: Each catalog can have its own storage location for physical separation.
*   **Governance**: Enables hierarchical permission flow (catalog → schema → table).

**Curiosity Check:**  
*Why not just use schemas?*  
Because catalogs allow **bigger isolation** (e.g., separate production vs development environments).

***

## **3. How It Works (Structure & Privileges)**

### **Hierarchy**

    Metastore (account level)
       └── Catalog
            └── Schema
                 ├── Table
                 ├── View
                 ├── Volume
                 ├── Function
                 └── Model

*   **Metastore**: Registered at account level; stores all catalogs.
*   **Catalog**: Logical unit for isolation and governance.
*   **Privileges**:
    *   **USE CATALOG** → Needed to access anything inside.
    *   **SELECT on catalog** → Read all tables inside.
    *   **CREATE TABLE on catalog** → Create tables in any schema inside.

***

## **4. How Should You Organize Catalogs?**

*   Mirror **organizational units** or **environment scopes**:
    *   Example:
        *   `prod_catalog` → Production data.
        *   `dev_catalog` → Development data.
        *   `customer_data` → Sensitive customer info.
*   Each catalog can have its **own managed storage location** for isolation.

***

## **5. Catalog Types**

*   **Standard Catalog**: Default type for organizing data.
*   **Foreign Catalog**: Used in **Lakehouse Federation** to represent external databases (e.g., Snowflake, MySQL).
*   **Special Catalogs**:
    *   `hive_metastore`: Legacy Hive metastore objects.
    *   **Workspace Catalog**: Auto-created for new workspaces; all users can access by default.
    *   **Default Catalog**: Pre-configured so you can run queries without specifying catalog name.

***

## **6. Workspace-Catalog Binding**

*   Lets you **restrict catalog access to specific workspaces**.
*   Example:
    *   `prod_catalog` → Accessible only from `prod_workspace`.

***

## **7. Practical Use Cases**

*   **Data Isolation**:
    *   Separate catalogs for **production vs development**.
*   **Security**:
    *   Grant **USE CATALOG** only to authorized teams.
*   **Cross-System Queries**:
    *   Use **Foreign Catalogs** for external databases.
*   **Governance**:
    *   Apply RBAC at catalog level for compliance.

***

## **8. Example SQL Commands**

### Create a Catalog:

```sql
CREATE CATALOG prod_catalog;
```

### Grant Access:

```sql
GRANT USE CATALOG ON CATALOG prod_catalog TO `data_engineers`;
```

***

## ✅ **Quick Visual Summary**

    Metastore
       ├── Catalog (prod_catalog, dev_catalog)
       │      └── Schema (sales, marketing)
       │           ├── Tables
       │           ├── Views
       │           ├── Volumes
       │           └── Models

***

### 🧠 **Final Thought**

Catalogs = **Your top-level data organizer + security gate**.  
Plan them carefully because they define **how your data is grouped, isolated, and governed**.

***



.


***

## ✅ **1. What Is a Catalog?**

A **catalog** is the **top-level container** in Unity Catalog.  
It organizes **schemas**, and each schema contains **tables, views, volumes, models, and functions**.

Think of it like:

    Catalog → Schema → Table/View/Volume

Example:

    finance_catalog.sales.orders

***

## ✅ **2. Why Does It Matter?**

*   **Data Organization**: Keeps your data structured.
*   **Security**: Catalog-level permissions control everything inside.
*   **Isolation**: Each catalog can have its own storage location.
*   **Governance**: Enables hierarchical permissions (catalog → schema → table).

***

## ✅ **3. Requirements Before You Create**

*   **Privileges**:
    *   Must be **Metastore Admin** or have `CREATE CATALOG` privilege.
*   **Compute**:
    *   Databricks Runtime **11.3+** with Unity Catalog access mode.
    *   SQL Warehouses always support Unity Catalog.
*   **Storage**:
    *   If no metastore-level storage exists, you **must specify a managed storage location**.
*   **Special Cases**:
    *   **Shared Catalog** → Requires Delta Sharing share and `USE PROVIDER` privilege.
    *   **Foreign Catalog** → Requires Databricks Runtime **13.1+** and a defined connection.

***

## ✅ **4. Types of Catalogs**

*   **Standard Catalog**: For organizing data and AI assets.
*   **Foreign Catalog**: Mirrors an external database (Snowflake, MySQL) for **Lakehouse Federation**.
*   **Shared Catalog**: Created from a **Delta Sharing share** for secure data sharing.

***

## ✅ **5. How to Create a Catalog**

You can use:

*   **Catalog Explorer (UI)**.
*   **SQL Commands**.
*   **REST API**.
*   **Databricks CLI**.
*   **Terraform**.

***

### **Option 1: Using Catalog Explorer (UI)**

1.  Go to **Data → Catalogs → Create Catalog**.
2.  Enter:
    *   **Catalog Name**.
    *   **Type** (Standard, Foreign, Shared).
3.  For **Standard Catalog**:
    *   (Optional but recommended) Specify **Managed Storage Location**.
4.  For **Foreign Catalog**:
    *   Select a **Connection** and provide **Database Name**.
5.  For **Shared Catalog**:
    *   Select **Provider** and **Share**.
6.  Click **Create**.
7.  Configure:
    *   **Workspace Binding** (limit access to specific workspaces).
    *   **Permissions** (grant privileges like `USE CATALOG`, `SELECT`).
    *   **Tags & Comments** for easy discovery.

***

### **Option 2: Using SQL**

#### **Standard Catalog**

```sql
CREATE CATALOG IF NOT EXISTS finance_catalog
MANAGED LOCATION 'abfss://my-container@storageaccount.dfs.core.windows.net/finance'
COMMENT 'Catalog for finance data';
```

#### **Shared Catalog**

```sql
CREATE CATALOG IF NOT EXISTS shared_catalog
USING SHARE provider_name.share_name
COMMENT 'Shared data catalog';
```

#### **Foreign Catalog**

```sql
CREATE FOREIGN CATALOG IF NOT EXISTS snowflake_catalog
USING CONNECTION snowflake_conn
OPTIONS (database 'sales_db');
```

***

## ✅ **6. After Creation**

*   Two schemas are auto-created:
    *   `default`
    *   `information_schema`
*   Assign **privileges**:
    ```sql
    GRANT USE CATALOG ON CATALOG finance_catalog TO `data_engineers`;
    ```
*   Add schemas and data objects:
    ```sql
    CREATE SCHEMA finance_catalog.sales;
    CREATE TABLE finance_catalog.sales.orders (...);
    ```

***

## ✅ **7. Practical Use Cases**

*   **Production vs Development Isolation**:
    *   `prod_catalog` vs `dev_catalog`.
*   **Secure Sharing**:
    *   Use **Shared Catalog** for partner collaboration.
*   **Cross-System Queries**:
    *   Use **Foreign Catalog** for Snowflake/MySQL.
*   **Governance**:
    *   Apply RBAC at catalog level for compliance.

***

### 🧠 **Curiosity Check**

*What happens if I don’t specify a managed storage location?*  
If your metastore doesn’t have a default storage location, **catalog creation will fail**.

***

## ✅ **Quick Visual Summary**

    Metastore
       ├── Catalog (Standard / Foreign / Shared)
       │      └── Schema
       │           ├── Tables
       │           ├── Views
       │           ├── Volumes
       │           └── Models

***

### **Final Thought**

Catalogs = **Your top-level data organizer + security gate**.  
Plan them carefully because they define **how your data is grouped, isolated, and governed**.

***




.

 difference between **default catalog** and **workspace catalog** in Databricks:

***

### ✅ **Default Catalog**

*   **What it is**:  
    The catalog that is automatically assigned when you create a Databricks workspace.
*   **Scope**:  
    Exists only within that workspace. It is **not shared across workspaces**.
*   **Governance**:  
    Uses **workspace-level permissions**, not Unity Catalog RBAC.
*   **Data Access**:  
    Tables and schemas created here are governed by **legacy workspace ACLs**, not centralized governance.
*   **Use Case**:  
    For older deployments or when Unity Catalog is not enabled.

***

### ✅ **Workspace Catalog**

*   **What it is**:  
    A catalog created and managed under **Unity Catalog**.
*   **Scope**:  
    Can be shared across multiple workspaces connected to the same **Unity Catalog metastore**.
*   **Governance**:  
    Uses **Unity Catalog RBAC** for fine-grained access control.
*   **Data Access**:  
    Tables, schemas, and volumes are governed centrally with **audit and compliance features**.
*   **Use Case**:  
    Recommended for modern deployments where centralized governance is required.

***

### **Key Differences Table**

| Feature            | Default Catalog  | Workspace Catalog (Unity Catalog) |
| ------------------ | ---------------- | --------------------------------- |
| Governance Model   | Workspace ACLs   | Unity Catalog RBAC                |
| Scope              | Single workspace | Multi-workspace (shared)          |
| Compliance & Audit | Limited          | Full audit & lineage              |
| Recommended For    | Legacy setups    | Enterprise governance             |

***

✅ **Bottom Line**:

*   **Default Catalog** = legacy, workspace-scoped, ACL-based.
*   **Workspace Catalog** = Unity Catalog-based, multi-workspace, RBAC-driven.

***




***

# ✅ **Manage Catalogs in Azure Databricks Unity Catalog**



## **1. Why Manage Catalogs?**

*   **Update ownership and permissions** for governance.
*   **Add tags and comments** for better discovery.
*   **Delete unused catalogs** to keep your environment clean.
*   **Control workspace bindings** for isolation.

***

## **2. Requirements**

*   Workspace must be linked to a **Unity Catalog metastore**.
*   Compute must use **Unity Catalog-compliant access mode** (SQL warehouses always support UC).
*   Permissions vary by task:
    *   **View** → `USE CATALOG` or `BROWSE`.
    *   **Update** → `MANAGE` + `USE CATALOG` or ownership.
    *   **Delete** → Catalog owner or `MANAGE` + `USE CATALOG`.

***

## **3. Tasks You Can Perform**

### ✅ **View Catalog Details**

*   **UI (Catalog Explorer)**:
    *   Go to **Data → Catalogs → Select Catalog**.
    *   Check tabs: **Schemas**, **Details**, **Permissions**, **Workspaces**.
*   **SQL**:
    ```sql
    SHOW CATALOGS;                -- List all catalogs
    DESCRIBE CATALOG finance;     -- View details
    DESCRIBE CATALOG EXTENDED finance; -- Full details
    ```

***

### ✅ **Update a Catalog**

You can:

*   Change **owner**.
*   Add or update **tags/comments**.
*   Grant/revoke **permissions**.
*   Add **schemas**.

**UI**:

*   Use **Overview tab** for owner, tags, comments.
*   Use **Permissions tab** for privileges.
*   Use **kebab menu** for rename.

**SQL**:

```sql
ALTER CATALOG finance SET OWNER TO `new_owner`;
ALTER CATALOG finance SET TAGS ('env'='prod');
GRANT USE CATALOG ON CATALOG finance TO `data_engineers`;
```

**Note**: Renaming via SQL requires creating a new catalog and moving assets.

***

### ✅ **Delete a Catalog**

*   **Warning**: Do NOT delete the **main catalog** (can break operations).
*   Must delete all schemas except `information_schema` first.

**UI**:

*   Go to **Catalog Explorer → Select Catalog → Kebab Menu → Delete**.

**SQL**:

```sql
DROP CATALOG finance CASCADE;   -- Deletes catalog and all schemas
DROP CATALOG finance RESTRICT;  -- Deletes only if empty
```

***

## **4. Practical Use Cases**

*   **Environment isolation**:
    *   Bind `prod_catalog` to `prod_workspace`.
*   **Governance**:
    *   Assign least privilege (e.g., `USE CATALOG` only).
*   **Cleanup**:
    *   Remove unused catalogs after migration.

***

## ✅ **Quick Visual Summary**

    Metastore
       ├── Catalog (Manage: View | Update | Delete)
       │      └── Schema
       │           ├── Tables
       │           ├── Views
       │           ├── Volumes

***

### 🧠 **Final Thought**

Managing catalogs = **controlling the top-level gate** for your data.  
Always follow **least privilege principle** and **bind catalogs to workspaces** for isolation.

***


.

 **“Manage the default catalog” in Unity Catalog**:

***

### **What is the default catalog?**

*   Each **workspace enabled for Unity Catalog** has a default catalog.
*   If you **don’t specify a catalog name** in your SQL or data operations, the system assumes the default catalog.
*   Example: `CREATE TABLE myschema.mytable` will use the default catalog automatically.

***

### **Where is it set and how does it apply?**

*   **Workspace Admin Settings UI**: Admins can view or change the default catalog.
*   **Cluster-level Spark config**: You can override the default catalog for a cluster using:
        spark.databricks.sql.initial.catalog.namespace
*   Applies only to compute resources compatible with Unity Catalog:
    *   SQL warehouses
    *   Clusters with **standard or dedicated access mode**
*   If compute is **not Unity Catalog-compatible**, the default is `hive_metastore`.

***

### **Order of catalog resolution**

When a command doesn’t specify a catalog, Databricks checks in this order:

1.  **Session setting**: `USE CATALOG` or JDBC setting.
2.  **Cluster Spark config**: `spark.databricks.sql.initial.catalog.namespace`.
3.  **Workspace default catalog**.

***

### **Special cases**

*   **Lakeflow Spark Declarative Pipelines**: Pipeline config overrides workspace default.
*   **Migration scenario**: You can keep `hive_metastore` as default to avoid breaking existing code.

***

### **Changing the default catalog**

*   **Workspace level**:
    1.  Log in as admin → **Admin Settings → Advanced tab**.
    2.  Enter new catalog name → **Save**.
    3.  Restart SQL warehouses/clusters for changes to apply.
*   **Cluster level**:
    *   Set Spark config:
            spark.databricks.sql.initial.catalog.namespace
    *   Not available for SQL warehouses.

**Warning**: Changing the default catalog can break existing queries or pipelines.

***

### **Viewing the current default catalog**

*   **Admin Settings**: Advanced tab → Default catalog row.
*   **SQL command**:
    ```sql
    SELECT current_catalog();
    ```
    Works if:
    *   No `USE CATALOG` or JDBC override in session.
    *   No cluster-level Spark config overriding it.

***

✅ This is the essence: **Default catalog = fallback catalog for operations without explicit catalog name. Configurable at workspace or cluster level, but changes can impact existing code.**



Here’s a clear explanation of **“Limit catalog access to specific workspaces” (workspace-catalog binding in Unity Catalog)**:

***

### **What is workspace-catalog binding?**

*   It’s a way to **restrict a catalog to one or more workspaces** in your Azure Databricks account.
*   **Default behavior**: All catalogs in a metastore are shared with all workspaces attached to that metastore.
*   **Exception**: The automatically created *workspace catalog* is bound only to its workspace unless you explicitly share it.

***

### **Why use it?**

Typical scenarios:

*   **Production isolation**: Only production workspaces can access production data.
*   **Sensitive data control**: Restrict access to dedicated workspaces.
*   **Read-only dev access**: Developers can view production data but cannot modify it.

***

### **Access modes**

*   **Read & Write**: Full access from the workspace.
*   **Read-only**: Blocks all write operations from that workspace.

***

### **How bindings work**

*   Workspace-catalog bindings override user-level grants.
*   Example:  
    If `prod_catalog` is bound to Prod workspaces only, a user with SELECT permission cannot access it from Dev workspace.
*   Reflected everywhere:
    *   **Information schema**: Shows only catalogs accessible in the current workspace.
    *   **Data lineage & search UIs**: Respect bindings.

***

### **How to bind a catalog to workspaces**

You need **Metastore admin**, **catalog owner**, or **MANAGE + USE CATALOG** permissions.

#### **Option 1: Catalog Explorer (UI)**

1.  Log in to a workspace linked to the metastore.
2.  Go to **Data → Catalog**.
3.  Select the catalog → **Workspaces tab**.
4.  Clear **All workspaces have access**.
5.  Click **Assign to workspaces** → choose workspaces.
6.  (Optional) Change access to **read-only** via **Manage Access Level**.
7.  To revoke: select workspace → click **Revoke**.

***

#### **Option 2: Databricks CLI**

Two steps:

1.  **Set isolation mode to ISOLATED**:
    ```bash
    databricks catalogs update <my-catalog> \
    --isolation-mode ISOLATED \
    --profile <profile-name>
    ```
    Default is `OPEN`.

2.  **Assign workspaces**:
    ```bash
    databricks workspace-bindings update-bindings catalog <my-catalog> \
    --json '{
      "add": [{"workspace_id": <workspace-id>, "binding_type": "<BINDING_TYPE_READ_WRITE or READ_ONLY>"}],
      "remove": [{"workspace_id": <workspace-id>, "binding_type": "<BINDING_TYPE_READ_WRITE or READ_ONLY>"}]
    }' --profile <profile-name>
    ```

To **list bindings**:

```bash
databricks workspace-bindings get-bindings catalog <my-catalog> \
--profile <profile-name>
```

***

### **Unbinding a catalog**

*   Use **Catalog Explorer** or CLI (`update-bindings` with `remove`).
*   **Important**: If you unbind the automatically created workspace catalog, you must manually grant permissions to workspace admins because they are workspace-local, not account-level.

***

✅ **Key takeaway**: Workspace-catalog binding is essential for **data isolation and security**. It ensures catalogs are only accessible from designated workspaces, regardless of user-level permissions.

***


Here’s a **clear and summarized explanation** combining both topics:

***

## ✅ **Part 1: Default Catalog vs Workspace Catalog**

*   **Default Catalog**
    *   The catalog Databricks uses when you don’t explicitly specify one in `catalog.schema.table`.
    *   Historically, this was **`hive_metastore`** (legacy).
    *   In Unity Catalog-enabled workspaces, the default is usually **`main`** or another UC catalog you set.
    *   Purpose: Avoid breaking old code that assumes no catalog prefix.

*   **Workspace Catalog**
    *   A catalog automatically created for each workspace when Unity Catalog is enabled.
    *   Bound only to that workspace unless you share it.
    *   Used for workspace-specific data isolation.

**Special Cases:**

*   **Lakeflow Pipelines**: They can override the workspace default by specifying `catalog` in pipeline config (e.g., `finance_catalog`). If not specified, they use the workspace default.
*   **Migration**: Keep `hive_metastore` as default temporarily so old scripts still work, then gradually move to UC catalogs.

***

## ✅ **Part 2: Workspace-Catalog Binding (Limit Catalog Access)**

*   **What is it?**  
    A feature in Unity Catalog that restricts which workspaces can access a catalog.
    *   Default: All catalogs in a metastore are shared with all workspaces.
    *   Exception: Workspace catalog is bound only to its workspace unless shared.

*   **Why use it?**
    *   **Production isolation**: Prod catalog → Prod workspace only.
    *   **Sensitive data control**: Restrict access to secure workspaces.
    *   **Read-only dev access**: Developers can view but not modify prod data.

*   **Access Modes:**
    *   **Read & Write**: Full access.
    *   **Read-only**: Blocks writes from that workspace.

*   **How it works:**
    *   Workspace binding overrides user-level grants.
    *   If `prod_catalog` is bound to Prod workspace only, even a user with SELECT cannot access it from Dev workspace.

*   **How to configure:**
    *   **UI (Catalog Explorer)**: Assign workspaces and set access level.
    *   **CLI**:
        *   Set isolation mode: `ISOLATED` vs `OPEN`.
        *   Update bindings with `READ_WRITE` or `READ_ONLY`.

**Key takeaway:**  
Workspace-catalog binding ensures catalogs are only accessible from designated workspaces, regardless of user permissions.

***

### ✅ **Visual Summary**

    Metastore
     ├── Catalogs (default: shared with all workspaces)
     │     ├── Workspace Catalog (bound to one workspace)
     │     ├── Prod Catalog (bound to prod workspace only)
     │     └── Dev Catalog (bound to dev workspace)

*   Default catalog = fallback for queries without catalog prefix.
*   Workspace catalog = auto-created for isolation.
*   Lakeflow can override default via config.
*   Workspace-catalog binding = enforce isolation at catalog level.

***

👉 Do you want me to **create a single diagram showing: Default catalog, workspace catalog, Lakeflow override, and workspace-catalog binding together**? Or a **cheat sheet with CLI + SQL commands for all these operations**?
